In [ ]:
import torch
import torch.nn as nn

Simple Model Architecture!

In [ ]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, features):
        output = self.linear(features)
        output = self.sigmoid(output)
        return output

Prepare the dataset!

In [ ]:
features = torch.rand(10,5)
features
len(features[0])

Initialize the Model!

In [ ]:
model = MyNN(len(features[0]))
model.forward(features)

# Inspect Model Weights and Bias!
model.linear.weight
model.linear.bias

In [ ]:
from torchinfo import summary

summary(model, input_size=(10, 5))

NEURAL NETWORK WITH HIDDEN LAYERS!

In [ ]:
class MyNN_hidden(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear1 = nn.Linear(num_features, 3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, features):
        output = self.linear1(features)
        output = self.relu(output)
        output = self.linear2(output)
        output = self.sigmoid(output)
        return output

In [ ]:
model = MyNN_hidden(len(features[0]))
model.forward(features)

model.linear1.weight
# model.linear1.bias

model.linear2.weight
# model.linear2.bias

APPLY NN ON CANCER DATASET!

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

X_train_tensor = torch.from_numpy(X_train).to(torch.float32)
X_test_tensor = torch.from_numpy(X_test).to(torch.float32)
y_train_tensor = torch.from_numpy(y_train).to(torch.float32)
y_test_tensor = torch.from_numpy(y_test).to(torch.float32)

In [ ]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, features):
        output = self.linear(features)
        output = self.sigmoid(output)
        return output

FULL TRAINING PIPELINE:

In [ ]:
# hyper-parameters
epochs = 50
lr = 1e-2

model = MyNN(len(X_train_tensor[0]))

# loss form NN
loss_fun = nn.BCELoss()
# optimiser from torch
optimiser = torch.optim.SGD(model.parameters(), lr)

In [ ]:
for epoch in range(epochs):
    # clear gradients
    optimiser.zero_grad()
    # forward pass
    y_pred = model.forward(X_train_tensor)
    # calculate the loss
    loss = loss_fun(y_pred, y_train_tensor.reshape(-1,1))
    # back propagation
    loss.backward()
    # parameter update
    optimiser.step()

    print(f"Epoch:{epoch} with loss:{loss}")

MODEL EVALUATION!

In [ ]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred>0.9).float()
    accuracy = (y_pred==y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy}")